In [1]:
# Cell 1: Imports and Basic Setup
import os
import sys

# Fix PROJ paths BEFORE any imports
# Use rasterio's bundled proj data since that's what's failing
PROJ_PATH = '/home/ejalilva/.conda/envs/terratorch-tune/lib/python3.12/site-packages/rasterio/proj_data'
os.environ['PROJ_LIB'] = PROJ_PATH
os.environ['PROJ_DATA'] = PROJ_PATH

# Also try pyproj's path as a backup
PYPROJ_PATH = '/home/ejalilva/.conda/envs/terratorch-tune/lib/python3.12/site-packages/pyproj/proj_dir/share/proj'
os.environ['PYPROJ_DATADIR'] = PYPROJ_PATH

print(f"Set PROJ_LIB to: {os.environ['PROJ_LIB']}")
print(f"Set PYPROJ_DATADIR to: {os.environ['PYPROJ_DATADIR']}")

# Verify the file exists
proj_db = os.path.join(PROJ_PATH, 'proj.db')
if os.path.exists(proj_db):
    print(f"✓ proj.db found at: {proj_db}")
else:
    print(f"✗ proj.db NOT found at: {proj_db}")

# Clean up any conflicting GEOSpyD paths
if 'PATH' in os.environ:
    path_parts = os.environ['PATH'].split(':')
    clean_path = [p for p in path_parts if 'GEOSpyD' not in p]
    os.environ['PATH'] = ':'.join(clean_path)
    
            
import numpy as np
import torch
import optuna
from optuna.integration import PyTorchLightningPruningCallback
from optuna.trial import TrialState
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Disable albumentations version check
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'

torch.set_float32_matmul_precision('medium')  # Enable Tensor Cores

# Check GPU availability
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs available: {num_gpus}")
else:
    print("No GPUs available, using CPU")

Set PROJ_LIB to: /home/ejalilva/.conda/envs/terratorch-tune/lib/python3.12/site-packages/rasterio/proj_data
Set PYPROJ_DATADIR to: /home/ejalilva/.conda/envs/terratorch-tune/lib/python3.12/site-packages/pyproj/proj_dir/share/proj
✓ proj.db found at: /home/ejalilva/.conda/envs/terratorch-tune/lib/python3.12/site-packages/rasterio/proj_data/proj.db
Number of GPUs available: 2


In [2]:
# Cell 2: Setup Terratorch Path and Import
# Get absolute path of the local package
local_package_path = os.path.abspath(os.path.join(os.getcwd(), '../..', 'terratorch'))

# Add the path to system path if it's not already there
if local_package_path not in sys.path:
    sys.path.insert(0, local_package_path)

# If you had previously imported terratorch, reload it
import importlib
if 'terratorch' in sys.modules:
    importlib.reload(sys.modules['terratorch'])

import terratorch
print(f"Terratorch loaded from: {terratorch.__file__}")

from terratorch.datamodules import MultiTemporalCropClassificationDataModule
from terratorch.datasets import MultiTemporalCropClassification
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datasets.transforms import FlattenTemporalIntoChannels, UnflattenTemporalFromChannels

Terratorch loaded from: /gpfsm/dnb33/ejalilva/dev/terratorch/terratorch/__init__.py


In [3]:
# Cell 3: Import Other Libraries
import albumentations as A
from albumentations.pytorch import ToTensorV2

import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping

# Import your custom callbacks
from loss_callback import LossTrackerCallback
from confusionMatrix_callback_withVal import ConfusionMatrixCallback


In [4]:
# Cell 4: Configuration
DATASET_PATH = '/discover/nobackup/ejalilva/data/prithvi/datasets--ibm-nasa-geospatial--multi-temporal-irrigation-classificaction/snapshots/04b439f179e52a7b144f69676210eecd30c39cfc/'
STUDY_NAME = f"prithvi_tuning_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
N_TRIALS = 2  # Reduced to 2 for testing
MAX_EPOCHS_TUNING = 3  # Reduced for testing
N_GPUS = 1

print(f"Dataset path: {DATASET_PATH}")
print(f"Study name: {STUDY_NAME}")
print(f"Will run {N_TRIALS} trials with {MAX_EPOCHS_TUNING} epochs each")

Dataset path: /discover/nobackup/ejalilva/data/prithvi/datasets--ibm-nasa-geospatial--multi-temporal-irrigation-classificaction/snapshots/04b439f179e52a7b144f69676210eecd30c39cfc/
Study name: prithvi_tuning_20251114_091056
Will run 2 trials with 3 epochs each


In [5]:
# Cell 5: Create Data Module (Test it first)
# Define transforms
transforms = [
    terratorch.datasets.transforms.FlattenTemporalIntoChannels(),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2(),
    terratorch.datasets.transforms.UnflattenTemporalFromChannels(n_timesteps=3),
]

# Create data module with fixed parameters for testing
test_datamodule = MultiTemporalCropClassificationDataModule(
    batch_size=16,
    data_root=DATASET_PATH,
    train_transform=transforms,
    val_transform=transforms,
    test_transform=transforms,
    reduce_zero_label=False,
    expand_temporal_dimension=True,
    use_metadata=True,
    num_workers=4  # Reduced for testing
)

# Test data loading
print("Testing data module setup...")
test_datamodule.setup("fit")
print(f"Train dataset size: {len(test_datamodule.train_dataset)}")
print(f"Val dataset size: {len(test_datamodule.val_dataset)}")
print("Data module works!")

Testing data module setup...
Train dataset size: 3058
Val dataset size: 766
Data module works!


In [6]:
# Cell 6: Test a Single Model Configuration (Before Optuna)
print("Testing a single model configuration...")

# Set seed
pl.seed_everything(42)

# Fixed hyperparameters for testing
test_lr = 1e-4
test_weight_decay = 0.1
test_head_dropout = 0.3
test_batch_size = 16
test_decoder_channels = 256

# Setup test directories
test_output_dir = os.path.join("test_run", "single_test")
os.makedirs(test_output_dir, exist_ok=True)

# Logger
test_logger = TensorBoardLogger(
    save_dir=test_output_dir,
    name="test_logs"
)

# Simple callbacks for testing
test_checkpoint = ModelCheckpoint(
    dirpath=os.path.join(test_output_dir, "checkpoints"),
    monitor="val/Multiclass_Jaccard_Index",
    mode="max",
    save_top_k=1
)

# Create trainer for testing
test_trainer = pl.Trainer(
    accelerator="auto",
    devices=N_GPUS,
    precision="bf16-mixed",
    logger=test_logger,
    max_epochs=2,  # Just 2 epochs for testing
    check_val_every_n_epoch=1,
    log_every_n_steps=10,
    enable_checkpointing=True,
    callbacks=[test_checkpoint],
    default_root_dir=test_output_dir,
    enable_model_summary=True,
    enable_progress_bar=True
)

print("Trainer created successfully")

[rank: 0] Seed set to 42
Using bfloat16 Automatic Mixed Precision (AMP)


Testing a single model configuration...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Trainer created successfully


In [7]:
# Cell 7: Create and Test Model
# Model arguments
model_args = {
    "decoder": "UperNetDecoder",
    "backbone_pretrained": True,
    "backbone": "prithvi_eo_v2_300_tl",
    "backbone_in_channels": 6,
    "backbone_features_only": True,
    "backbone_coords_encoding": ["time", "location"],
    "rescale": True,
    "backbone_bands": ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"],
    "backbone_num_frames": 3,
    "num_classes": 4,
    "head_dropout": test_head_dropout,
    "decoder_channels": test_decoder_channels,
    "decoder_scale_modules": True,
    "necks": [
        {
            "name": "SelectIndices",
            "indices": [5, 11, 17, 23]
        },
        {
            "name": "ReshapeTokensToImage",
            "effective_time_dim": 3
        }
    ]
}

# Create model
test_model = SemanticSegmentationTask(
    model_args=model_args,
    plot_on_val=False,
    class_weights=[2.28, 1.02, 1.37, 0.54],
    loss="ce",
    lr=test_lr,
    optimizer="AdamW",
    optimizer_hparams={"weight_decay": test_weight_decay},
    ignore_index=-1,
    freeze_backbone=False,
    freeze_decoder=False,
    model_factory="EncoderDecoderFactory",
)

print("Model created successfully")
print(f"Model type: {type(test_model)}")

INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (('ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL', 'Prithvi_EO_V2_300M_TL.pt'))
INFO:root:Loaded weights for HLSBands.BLUE in position 0 of patch embed
INFO:root:Loaded weights for HLSBands.GREEN in position 1 of patch embed
INFO:root:Loaded weights for HLSBands.RED in position 2 of patch embed
INFO:root:Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_1 in position 4 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_2 in position 5 of patch embed


Model created successfully
Model type: <class 'terratorch.tasks.segmentation_tasks.SemanticSegmentationTask'>


In [8]:
# Cell 8: Run Quick Training Test
print("Starting quick training test...")
try:
    test_trainer.fit(test_model, datamodule=test_datamodule)
    print("Training test completed successfully!")
    
    # Get metrics
    if "val/Multiclass_Jaccard_Index" in test_trainer.callback_metrics:
        val_score = test_trainer.callback_metrics["val/Multiclass_Jaccard_Index"].item()
        print(f"Validation Jaccard Index: {val_score:.4f}")
except Exception as e:
    print(f"Error during training: {e}")
    import traceback
    traceback.print_exc()

Starting quick training test...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | PixelWiseModel   | 364 M  | train
1 | criterion     | CrossEntropyLoss | 0      | train
2 | train_metrics | MetricCollection | 0      | train
3 | val_metrics   | MetricCollection | 0      | train
4 | test_metrics  | ModuleList       | 0      | train
-----------------------------------------------------------
364 M     Trainable params
0         Non-trainable params
364 M     Total params
1,457.823 Total estimated model params size (MB)
647       Modules in train mode
0         Modules in eval mode
SLURM auto-requeueing enabled. Setting signal handlers.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.


Training test completed successfully!
Validation Jaccard Index: 0.4487


In [9]:
# Cell 9: Now Test Optuna with Single Trial
print("\nTesting Optuna with a single trial...")

# Create a simple objective function for testing
def test_objective(trial):
    """Simplified objective for testing."""
    
    # Sample hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 0.01, 0.5, log=True)
    head_dropout = trial.suggest_float('head_dropout', 0.0, 0.5)
    batch_size = trial.suggest_categorical('batch_size', [8, 16])
    
    print(f"\nTrial {trial.number} hyperparameters:")
    print(f"  lr: {lr:.6f}")
    print(f"  weight_decay: {weight_decay:.4f}")
    print(f"  head_dropout: {head_dropout:.4f}")
    print(f"  batch_size: {batch_size}")
    
    # Set seed
    pl.seed_everything(42 + trial.number)
    
    # Create run name
    run_name = f"trial_{trial.number:03d}"
    output_dir = os.path.join("optuna_test", run_name)
    os.makedirs(output_dir, exist_ok=True)
    
    # Quick trainer for testing
    trainer = pl.Trainer(
        accelerator="auto",
        devices=N_GPUS,
        precision="bf16-mixed",
        max_epochs=2,  # Just 2 epochs for testing
        enable_checkpointing=False,
        enable_model_summary=False,
        enable_progress_bar=True,
        default_root_dir=output_dir
    )
    
    try:
        # Simple data module
        data_module = MultiTemporalCropClassificationDataModule(
            batch_size=batch_size,
            data_root=DATASET_PATH,
            train_transform=transforms,
            val_transform=transforms,
            test_transform=transforms,
            reduce_zero_label=False,
            expand_temporal_dimension=True,
            use_metadata=True,
            num_workers=4
        )
        
        # Simple model
        model = SemanticSegmentationTask(
            model_args=model_args,
            plot_on_val=False,
            class_weights=[2.28, 1.02, 1.37, 0.54],
            loss="ce",
            lr=lr,
            optimizer="AdamW",
            optimizer_hparams={"weight_decay": weight_decay},
            ignore_index=-1,
            freeze_backbone=False,
            freeze_decoder=False,
            model_factory="EncoderDecoderFactory",
        )
        
        # Train
        trainer.fit(model, datamodule=data_module)
        
        # Return a dummy score for testing
        return np.random.random()  # Random score for testing
        
    except Exception as e:
        print(f"Trial failed: {e}")
        return 0.0

# Create and run study with 1 trial
test_study = optuna.create_study(
    study_name="test_study",
    direction='maximize'
)

test_study.optimize(test_objective, n_trials=1)
print(f"\nOptuna test completed!")
print(f"Best value: {test_study.best_value:.4f}")
print(f"Best params: {test_study.best_params}")

[I 2025-11-13 21:09:25,688] A new study created in memory with name: test_study
[rank: 0] Seed set to 42
Using bfloat16 Automatic Mixed Precision (AMP)



Testing Optuna with a single trial...

Trial 0 hyperparameters:
  lr: 0.000026
  weight_decay: 0.2494
  head_dropout: 0.1458
  batch_size: 8


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (('ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL', 'Prithvi_EO_V2_300M_TL.pt'))
INFO:root:Loaded weights for HLSBands.BLUE in position 0 of patch embed
INFO:root:Loaded weights for HLSBands.GREEN in position 1 of patch embed
INFO:root:Loaded weights for HLSBands.RED in position 2 of patch embed
INFO:root:Loaded weights for HLSBands.NIR_NARROW in position 3 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_1 in position 4 of patch embed
INFO:root:Loaded weights for HLSBands.SWIR_2 in position 5 of patch embed
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=2` reached.
[I 2025-11-13 21:12:35,119] Trial 0 finished with value: 0.3745401188473625 and parameters: {'lr': 2.61333623306585e-05, 'weight_decay': 0.24944436805959586, 'head_dropout': 0.14578081747924915, 'batch_size': 8}. Best is trial 0 with value: 0.3745401188473625.



Optuna test completed!
Best value: 0.3745
Best params: {'lr': 2.61333623306585e-05, 'weight_decay': 0.24944436805959586, 'head_dropout': 0.14578081747924915, 'batch_size': 8}


In [ ]:
# Cell 10: If Everything Works, Run Full Optuna Study
print("\nIf all tests passed, you can now run the full study:")
print("1. Increase N_TRIALS to desired number (e.g., 50)")
print("2. Increase MAX_EPOCHS_TUNING to desired number (e.g., 20)")
print("3. Use the full objective function from original script")
print("\nCurrent configuration:")
print(f"  N_TRIALS: {N_TRIALS}")
print(f"  MAX_EPOCHS_TUNING: {MAX_EPOCHS_TUNING}")

In [ ]:
# Cell 11: Check Results
# After running, you can check the results
if os.path.exists("optuna_test"):
    print("Test outputs created:")
    for root, dirs, files in os.walk("optuna_test"):
        level = root.replace("optuna_test", "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files
            print(f"{subindent}{file}")